In [150]:
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

SEQ_MIN = 9
SEQ_MAX = 10
NET = nn.LSTM


Let's build our vocabulary, like before

In [151]:
base_tokens = [
    'red', 'blue', 'green', 'yellow',
    'dog', 'cat', 'bird', 'fish',
    'runs', 'jumps', 'swims', 'sleeps'
]

special_tokens = ['<PAD>', '<SOS>', '<EOS>']
vocab = special_tokens + base_tokens

word_to_id = {word: i for i, word in enumerate(vocab)}
id_to_word = {i: word for word, i in word_to_id.items()}

pad_id = word_to_id['<PAD>']
sos_id = word_to_id['<SOS>']
eos_id = word_to_id['<EOS>']
vocab_size = len(vocab)

print('vocab size:', vocab_size)
print(word_to_id)



vocab size: 15
{'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, 'red': 3, 'blue': 4, 'green': 5, 'yellow': 6, 'dog': 7, 'cat': 8, 'bird': 9, 'fish': 10, 'runs': 11, 'jumps': 12, 'swims': 13, 'sleeps': 14}


let's generate our dataset and convert things to input/output tensors

In [152]:
def make_reverse_examples(num_examples=1000, min_len=SEQ_MIN, max_len=SEQ_MAX):
    inputs = []
    targets = []

    for _ in range(num_examples):
        seq_len = random.randint(min_len, max_len)
        tokens = random.sample(base_tokens, seq_len)
        reversed_tokens = list(reversed(tokens))
        inputs.append(tokens)
        targets.append(reversed_tokens)

    return inputs, targets


input_sequences, target_sequences = make_reverse_examples()
print(input_sequences[:3])
print(target_sequences[:3])

max_input_len = max(len(seq) for seq in input_sequences)
max_target_len = max(len(seq) for seq in target_sequences) + 1  # room for <EOS>


def encode_input(seq):
    ids = [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_input_len - len(ids))
    return ids


def encode_decoder_input(seq):
    ids = [sos_id] + [word_to_id[word] for word in seq]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


def encode_decoder_target(seq):
    ids = [word_to_id[word] for word in seq] + [eos_id]
    ids = ids + [pad_id] * (max_target_len - len(ids))
    return ids


encoder_inputs = torch.tensor([encode_input(seq) for seq in input_sequences], dtype=torch.long)
decoder_inputs = torch.tensor([encode_decoder_input(seq) for seq in target_sequences], dtype=torch.long)
decoder_targets = torch.tensor([encode_decoder_target(seq) for seq in target_sequences], dtype=torch.long)

print('encoder_inputs shape:', encoder_inputs.shape)
print('decoder_inputs shape:', decoder_inputs.shape)
print('decoder_targets shape:', decoder_targets.shape)
print('example encoder input:', encoder_inputs[0])
print('example decoder input:', decoder_inputs[0])
print('example decoder target:', decoder_targets[0])


[['cat', 'swims', 'blue', 'runs', 'fish', 'jumps', 'sleeps', 'red', 'yellow'], ['bird', 'blue', 'fish', 'swims', 'red', 'green', 'yellow', 'runs', 'sleeps', 'dog'], ['cat', 'swims', 'jumps', 'blue', 'sleeps', 'fish', 'bird', 'yellow', 'dog']]
[['yellow', 'red', 'sleeps', 'jumps', 'fish', 'runs', 'blue', 'swims', 'cat'], ['dog', 'sleeps', 'runs', 'yellow', 'green', 'red', 'swims', 'fish', 'blue', 'bird'], ['dog', 'yellow', 'bird', 'fish', 'sleeps', 'blue', 'jumps', 'swims', 'cat']]
encoder_inputs shape: torch.Size([1000, 10])
decoder_inputs shape: torch.Size([1000, 11])
decoder_targets shape: torch.Size([1000, 11])
example encoder input: tensor([ 8, 13,  4, 11, 10, 12, 14,  3,  6,  0])
example decoder input: tensor([ 1,  6,  3, 14, 12, 10, 11,  4, 13,  8,  0])
example decoder target: tensor([ 6,  3, 14, 12, 10, 11,  4, 13,  8,  2,  0])


And let's make a DataLoader & test/train split

In [153]:
num_examples = len(encoder_inputs)
indices = list(range(num_examples))
random.shuffle(indices)

split = int(0.8 * num_examples)
train_idx = indices[:split]
test_idx = indices[split:]

train_dataset = TensorDataset(
    encoder_inputs[train_idx],
    decoder_inputs[train_idx],
    decoder_targets[train_idx],
)

# we're not actually going to validate with this dataset today
test_dataset = TensorDataset(
    encoder_inputs[test_idx],
    decoder_inputs[test_idx],
    decoder_targets[test_idx],
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

print('train examples:', len(train_dataset))
print('test examples:', len(test_dataset))


train examples: 800
test examples: 200


Now let's define out encoder/decoder model

In [154]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.net = NET(embedding_dim, hidden_dim, batch_first=True)

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        outputs, hidden = self.net(emb)
        return outputs, hidden


class Decoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_id)
        self.net = NET(embedding_dim, hidden_dim, batch_first=True)
        self.output = nn.Linear(hidden_dim, vocab_size)

    def forward(self, decoder_input_ids, hidden):
        emb = self.embedding(decoder_input_ids)
        outputs, hidden = self.net(emb, hidden)
        logits = self.output(outputs)
        return logits, hidden


class Seq2SeqNetwork(nn.Module):
    def __init__(self, vocab_size, embedding_dim=32, hidden_dim=64):
        super().__init__()
        self.encoder = Encoder(vocab_size, embedding_dim, hidden_dim)
        self.decoder = Decoder(vocab_size, embedding_dim, hidden_dim)

    def forward(self, encoder_input_ids, decoder_input_ids):
        encoder_outputs, hidden = self.encoder(encoder_input_ids)
        logits, hidden = self.decoder(decoder_input_ids, hidden)
        return logits

set up the training loop

In [155]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

model = Seq2SeqNetwork(vocab_size).to(device)
loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
optimizer = optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    for enc_batch, dec_in_batch, dec_tgt_batch in train_loader:
        enc_batch = enc_batch.to(device)
        dec_in_batch = dec_in_batch.to(device)
        dec_tgt_batch = dec_tgt_batch.to(device) # [batch_size, target_len]

        logits = model(enc_batch, dec_in_batch)  # [batch_size, target_len, vocab_size]

        # logits are a 3D tensor now, so we need to reshape it for CrossEntropyLoss which wants a 2D tensor
        # reshape with -1 keeps the last dimension as vocab_size and flattens the rest
        logits_2D = logits.reshape(-1, vocab_size) # [batch_size * target_len, vocab_size]
        dec_tgt_batch_1D = dec_tgt_batch.reshape(-1)

        loss = loss_fn(logits_2D, dec_tgt_batch_1D )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        non_pad_tokens = (dec_tgt_batch != pad_id).sum().item() # count the number of non-pad tokens
        total_loss += loss.item() * non_pad_tokens
        total_tokens += non_pad_tokens

    avg_loss = total_loss / total_tokens
    if (epoch + 1) % 5 == 0:
        print(f'epoch {epoch+1}, avg token loss={avg_loss:.4f}')


Using device: cpu
epoch 5, avg token loss=0.7970
epoch 10, avg token loss=0.2585
epoch 15, avg token loss=0.1481
epoch 20, avg token loss=0.1452
epoch 25, avg token loss=0.0358
epoch 30, avg token loss=0.0136


Now let's set up our decoder

In [156]:
def decode_greedy(input_tokens, model, max_steps=max_target_len):
    # some layers behave differently during training/inference like Dropout layers
    # this is probably not doing anything in this example
    model.eval()

    encoder_ids = encode_input(input_tokens)
    encoder_tensor = torch.tensor([encoder_ids], dtype=torch.long).to(device)

    # .no_grad() tells PyTorch not to track gradients
    with torch.no_grad():

        # we only use the encoder part of the model!
        encoder_outputs, hidden = model.encoder(encoder_tensor)

        # Now to build the decoder input - we don't have a teacher forcing example to use
        # so let's start with the start-of-sequence token
        start_token_id = sos_id
        
        # Make a batch with one sequence containing just that one token
        # Shape: [batch_size=1, sequence_length=1]
        decoder_input = [[start_token_id]]
        
        decoder_input = torch.tensor(decoder_input, dtype=torch.long)
        
        decoder_input = decoder_input.to(device)
        
        generated_ids = []

        for _ in range(max_steps):
            logits, hidden = model.decoder(decoder_input, hidden)
            
            # Shape goes from [batch_size, sequence_length, vocab_size]
            # to [batch_size, vocab_size] and we only want the one from the last time step
            last_step_logits = logits[:, -1, :]
            
            # Pick the vocabulary item with the highest score for each example in the batch
            # Shape: [batch_size]
            predicted_token_ids = last_step_logits.argmax(dim=1)
            
            # Since we only have one example in the batch, pull out that one token id as a Python number
            next_id = predicted_token_ids.item()

            if next_id == eos_id:
                break

            # prepare the just-predicted token as the decoder input on the next step
            generated_ids.append(next_id)
            decoder_input = torch.tensor([[next_id]], dtype=torch.long).to(device)

    return [id_to_word[i] for i in generated_ids]


and test it out

In [157]:
def print_list_diff(target: list[str], prediction: list[str]):
    mismatched_indices = set()
    for i, vals in enumerate(zip(target, prediction)):
        if vals[0] != vals[1]:
            mismatched_indices.add(i)
    
    for i, s in enumerate(target):
        pad_len = max(len(target[i]), len(prediction[i])) + 4
        if i in mismatched_indices:
            print('\033[1;32m' + s.ljust(pad_len) + '\033[0m', end='')
        else:
            print(s.ljust(pad_len), end='')
    
    print()

    for i, s in enumerate(prediction):
        pad_len = max(len(target[i]), len(prediction[i])) + 4
        if i in mismatched_indices:
            print('\033[1;31m' + s.ljust(pad_len) + '\033[0m', end='')
        else:
            print(s.ljust(pad_len), end='')

    print()



for i in range(5):
    source = input_sequences[test_idx[i]]
    target = target_sequences[test_idx[i]]
    prediction = decode_greedy(source, model)
    print('source:    ', source)
    if prediction == target:
        print('predicted correctly: ', prediction)
    else:
        print_list_diff(target, prediction)
    print()


source:     ['cat', 'blue', 'yellow', 'runs', 'dog', 'fish', 'sleeps', 'bird', 'red', 'jumps']
predicted correctly:  ['jumps', 'red', 'bird', 'sleeps', 'fish', 'dog', 'runs', 'yellow', 'blue', 'cat']

source:     ['blue', 'jumps', 'cat', 'dog', 'bird', 'runs', 'sleeps', 'green', 'red', 'swims']
swims    red    green    sleeps    runs    bird    dog    cat    jumps    blue     
swims    red    green    sleeps    runs    bird    dog    cat    blue     jumps    

source:     ['fish', 'cat', 'sleeps', 'green', 'red', 'yellow', 'swims', 'dog', 'bird']
predicted correctly:  ['bird', 'dog', 'swims', 'yellow', 'red', 'green', 'sleeps', 'cat', 'fish']

source:     ['red', 'jumps', 'swims', 'bird', 'green', 'runs', 'blue', 'dog', 'yellow', 'sleeps']
predicted correctly:  ['sleeps', 'yellow', 'dog', 'blue', 'runs', 'green', 'bird', 'swims', 'jumps', 'red']

source:     ['runs', 'green', 'yellow', 'red', 'sleeps', 'blue', 'cat', 'bird', 'fish']
predicted correctly:  ['fish', 'bird', 'cat', 'blue',

### results
GRU started to struggle a lot around 6-7 sequence lengths

LSTM did a lot better, almost perfect around 6-7 sequence lengths. it started to struggle a lot more around 10-11 sequence lengths